# Chronos-2 Quantisation

In [1]:
import os
from pathlib import Path

import torch

os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

from chop.models import get_model
from chop.ir.graph import MaseGraph
from chop.passes.graph import PASSES


c:\University\yr4\AdvancedDLSystems\Coursework\mase\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\University\yr4\AdvancedDLSystems\Coursework\mase\.venv\Lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
# Basic run config
OUTPUT_DIR = Path("artifacts")
GRAPH_NAME = "chronos2_quantized"
DEVICE = "cpu"
WRITE_QUANT_SUMMARY = False  # Set True to generate comparison CSVs (slower).

# Conservative starting quantisation config (only core arithmetic ops)
QUANT_PASS_ARGS = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
    "matmul": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
        }
    },
    "add": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 4,
            "weight_width": 8,
            "weight_frac_width": 4,
        }
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# 1) Load model
model = get_model("chronos-2", pretrained=True)
model.eval()
if DEVICE:
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but torch.cuda.is_available() is False.")
    model = model.to(DEVICE)

# Force eager attention so FX sees matmul/add/softmax nodes.
if hasattr(model.config, "_attn_implementation"):
    model.config._attn_implementation = "eager"

print("Loaded Chronos-2 model")

Loaded Chronos-2 model


In [4]:
# 2) Build graph + required metadata
mg = MaseGraph(
    model,
    hf_input_names=["context", "context_mask", "group_ids"],
)

batch_size = 1
c_len = model.config.chronos_config["context_length"]
dummy_in = {
    "context": torch.randn((batch_size, c_len), device=DEVICE),
    "context_mask": torch.ones((batch_size, c_len), dtype=torch.bool, device=DEVICE),
    "group_ids": torch.zeros((batch_size,), dtype=torch.long, device=DEVICE),
}

mg, _ = PASSES["init_metadata"](mg)
mg, _ = PASSES["add_common_metadata"](mg, pass_args={"dummy_in": dummy_in})

print(f"Graph nodes before quantisation: {sum(1 for _ in mg.nodes)}")

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[True, True, True,  ..., True, True, True]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7116,  0.9027,  0.4911,  ...,  1.1548,  0.4711, -0.0055]])
tensor([[2.9296e+00, 8.1495e-01, 2.4113e-01,  ..., 1.3335e+00, 2.2195e-01, 3.0386e-05]])
tensor([[1.0263]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 

In [ ]:
# 3) Quantise
orig_mg = None
if WRITE_QUANT_SUMMARY:
    # Needed only for before/after comparison in summarize_quantization.
    orig_mg = MaseGraph(model, hf_input_names=["context", "context_mask", "group_ids"])
    orig_mg, _ = PASSES["init_metadata"](orig_mg)
    orig_mg, _ = PASSES["add_common_metadata"](orig_mg, pass_args={"dummy_in": dummy_in})

mg, _ = PASSES["quantize"](mg, pass_args=QUANT_PASS_ARGS)
print("Quantisation pass complete")

if WRITE_QUANT_SUMMARY:
    PASSES["summarize_quantization"](
        mg,
        {"save_dir": OUTPUT_DIR / "quantize_summary", "original_graph": orig_mg},
    )
    print("Saved quantization summary to artifacts/quantize_summary")


tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[True, True, True,  ..., True, True, True]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7207,  0.8937,  0.4820,  ...,  1.1457,  0.4621, -0.0146]])
tensor([[-1.7116,  0.9027,  0.4911,  ...,  1.1548,  0.4711, -0.0055]])
tensor([[2.9296e+00, 8.1495e-01, 2.4113e-01,  ..., 1.3335e+00, 2.2195e-01, 3.0386e-05]])
tensor([[1.0263]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[-1.2955,  0.8022,  0.4675,  ...,  0.9769,  0.4497, -0.0054]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[1., 1., 1.,  ..., 1., 1., 1.]])
tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 

In [ ]:
# 4) Export quantised graph
base = OUTPUT_DIR / GRAPH_NAME
mg.export(str(base))
print(f"Exported: {base}.pt and {base}.mz")